# Model Deployment

In [1]:
%reset -f
import sys
for module in list(sys.modules.keys()):
  if module.startswith(("models", "assets", "data", "custom_datasets", "image_classification")):
    del sys.modules[module]
import torch
if torch.cuda.is_available():
  torch.cuda.empty_cache()
!rm -rf /content/*

In [2]:
import matplotlib.pyplot as plt
from torch import nn
import torchvision
from torchvision import transforms
import os
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
  from torchinfo import summary
except:
  print("Torchinfo not found! Installing...")
  %pip install -q torchinfo

!git clone https://github.com/syedhussain-dev/PyTorch
!mv PyTorch/image_classification .
!mv PyTorch/models/VisionTransformer .
!mv PyTorch/custom_datasets .
!rm -rf PyTorch
import image_classification.data_setup as data_setup
import image_classification.engine as engine
import image_classification.utils as utils
import image_classification.predictions as predictions

Torchinfo not found! Installing...
Cloning into 'PyTorch'...
remote: Enumerating objects: 717, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 717 (delta 153), reused 148 (delta 67), pack-reused 474 (from 1)
Receiving objects: 100% (717/717), 28.47 MiB | 19.79 MiB/s, done.
Resolving deltas: 100% (418/418), done.


In [3]:
extract_path = utils.download_data(
    source="https://github.com/syedhussain-dev/PyTorch/raw/refs/heads/main/data/images/watch_shoe_fragrance.zip",
    destination="watch_shoe_fragrance",
    remove_source=True
)

Unzipping


In [4]:
weights = torchvision.models.ViT_B_16_Weights.DEFAULT
auto_transform = weights.transforms()

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    )
])
train_dataloader, val_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=extract_path/"train",
    test_dir=extract_path/"test",
    val_dir=extract_path/"val",
    train_transform=auto_transform,
    test_and_val_transform=test_transform,
    batch_size=32
)
print(class_names)

['fragrance', 'shoe', 'watch']


In [5]:
from VisionTransformer.VisionTransformer import ViT
from torchinfo import summary
model0 = ViT().load_pretrained()
model0.to(device)
summary(
    model=model0,
    input_size=(1,3,224,224),
    col_names=["input_size","output_size","num_params","trainable"],
    col_width=20,
    row_settings=["var_names"]
)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 85.0MB/s]


Layer (type (var_name))                            Input Shape          Output Shape         Param #              Trainable
ViT (ViT)                                          [1, 3, 224, 224]     [1, 3]               152,064              True
├─PatchEmbedding (patch_embedding)                 [1, 3, 224, 224]     [1, 196, 768]        --                   True
│    └─Conv2d (create_patches)                     [1, 3, 224, 224]     [1, 768, 14, 14]     590,592              True
├─Dropout (emb_dropout)                            [1, 197, 768]        [1, 197, 768]        --                   --
├─ModuleList (encoder_blocks)                      --                   --                   --                   True
│    └─TransformEncoderBlock (0)                   [1, 197, 768]        [1, 197, 768]        --                   True
│    │    └─LayerNorm (norm1)                      [1, 197, 768]        [1, 197, 768]        1,536                True
│    │    └─MultiHeadSelfAttention (attention

In [6]:
writer0 = utils.create_writer(
    experiment_name="watch_shoe_fragrance",
    model_name="ViT_B_16",
    extra="30_epochs"
)

[INFO] Created SummaryWriter, saving to: runs/24-09-2026/watch_shoe_fragrance/ViT_B_16/10_epochs...


In [7]:
epochs = 30
optimiser0 = torch.optim.AdamW(params=model0.parameters(), lr=1e-5, weight_decay=0.01)
engine.batch_train(
    model0, train_dataloader, val_dataloader, test_dataloader, epochs, 1, device, False,
    optimiser0, torch.nn.CrossEntropyLoss(), writer0, torch.optim.lr_scheduler.CosineAnnealingLR(optimiser0, epochs, 1e-6), "vit_b_16.pth", "saved_models"
)

Epoch: 1 
 ---------------------------------------------------------

Train Loss: 1.09688 | Train Accuracy: 38.00% | Validation Loss: 1.07079 | Validation Accuracy: 50.00% | Time of Epoch: 40.29

Epoch: 2 
 ---------------------------------------------------------

Train Loss: 1.00227 | Train Accuracy: 60.17% | Validation Loss: 0.89594 | Validation Accuracy: 76.67% | Time of Epoch: 43.18

Epoch: 3 
 ---------------------------------------------------------

Train Loss: 0.61335 | Train Accuracy: 78.33% | Validation Loss: 0.72812 | Validation Accuracy: 68.00% | Time of Epoch: 45.04

Epoch: 4 
 ---------------------------------------------------------

Train Loss: 0.31838 | Train Accuracy: 90.50% | Validation Loss: 0.70451 | Validation Accuracy: 68.00% | Time of Epoch: 44.05

Epoch: 5 
 ---------------------------------------------------------

Train Loss: 0.25004 | Train Accuracy: 90.50% | Validation Loss: 0.57137 | Validation Accuracy: 79.33% | Time of Epoch: 44.39

Epoch: 6 
 ---------

In [8]:
list_of_preds = predictions.pred_and_store(extract_path/"test", model0, auto_transform, class_names, device)
from pprint import pprint
pprint(list_of_preds[:5])

[{'class_name': 'fragrance',
  'correct': True,
  'image_path': PosixPath('data/watch_shoe_fragrance/test/fragrance/29_Amouage_Interlude Black Iris Eau de Parfum.jpg'),
  'pred_class': 'fragrance',
  'pred_prob': 0.9895,
  'time_for_pred': 0.0608},
 {'class_name': 'fragrance',
  'correct': True,
  'image_path': PosixPath('data/watch_shoe_fragrance/test/fragrance/77_Xerjoff_Golden Moka Eau de Parfum.jpg'),
  'pred_class': 'fragrance',
  'pred_prob': 0.9994,
  'time_for_pred': 0.0215},
 {'class_name': 'fragrance',
  'correct': True,
  'image_path': PosixPath('data/watch_shoe_fragrance/test/fragrance/86_Anthologie de Grands Crus_Ambre Stone Eau de Parfum.jpg'),
  'pred_class': 'fragrance',
  'pred_prob': 0.999,
  'time_for_pred': 0.0198},
 {'class_name': 'fragrance',
  'correct': True,
  'image_path': PosixPath('data/watch_shoe_fragrance/test/fragrance/68_CALAJ Perfumes_Black Beard Extrait de Parfum.jpg'),
  'pred_class': 'fragrance',
  'pred_prob': 0.9997,
  'time_for_pred': 0.0207},
 {'

In [9]:
import gradio as gr
gr.__version__

'6.26.0'

In [10]:
model0.to("cpu")
next(iter(model0.parameters())).device

device(type='cpu')

In [11]:
from typing import Tuple, Dict
from timeit import default_timer as timer

def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken.
    """
    # Start the timer
    start_time = timer()

    # Transform the target image and add a batch dimension
    img = auto_transform(img).unsqueeze(0)

    # Put model into evaluation mode and turn on inference mode
    model0.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(model0(img), dim=1)

    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}

    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)

    # Return the prediction dictionary and prediction time
    return pred_labels_and_probs, pred_time

In [12]:
try:
  model0.load_state_dict(torch.load(f="saved_models/vit_b_16.pth", map_location=device))
except FileNotFoundError:
  print("Could not find file!")

In [13]:
title = "Watch Shoe Fragrance Determinator!"
description = "Using an Custom VisionTransformer B16 model"
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Label(num_top_classes=3, label="Predictions"), gr.Number(label="Prediction Time (s)")],
    title=title, description=description
)
demo.launch(debug=False, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0c94f928edcd5592ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# import shutil
# from pathlib import Path
# import random
# wsf_path = Path("demo/watch_shoe_fragrance")
# if wsf_path.exists():
#   shutil.rmtree(wsf_path)
# wsf_path.mkdir(parents=True, exist_ok=True)
# wsf_examples_path = wsf_path / "examples"
# wsf_examples_path.mkdir(parents=True, exist_ok=True)
# examples = [Path(filepath) for filepath in random.sample(list(Path(extract_path/"test").glob("*/*.jpg")),k=3)]
# for example in examples:
#   destination = wsf_examples_path / example.name
#   print(f"Copying {example} to {destination}")
#   shutil.copy2(src=example, dst=destination)
# shutil.copy2(src="models/effnetb2.pth",dst=wsf_path)

In [15]:
# %%writefile demo/watch_shoe_fragrance/model.py
# import torch
# import torchvision
# from torch import nn
# def create_effnetb2_model(seed:int=32,num_classes:int=3):
#   weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
#   transform = weights.transforms()
#   model = torchvision.models.efficientnet_b2(weights=weights)

#   for param in model.features.parameters():
#     param.requires_grad = False

#   for param in model.features[-2].parameters():
#     param.requires_grad = True

#   torch.manual_seed(seed)
#   model.classifier = nn.Sequential(
#       nn.Dropout(p=0.3,inplace=True),
#       nn.Linear(in_features=model.classifier[1].in_features,out_features=num_classes)
#   )

#   return model, transform


In [16]:
# %%writefile demo/watch_shoe_fragrance/app.py
# import gradio as gr
# import torch
# import os
# from model import create_effnetb2_model
# from typing import Tuple, Dict
# from timeit import default_timer as timer

# class_names = ["fragrance", "shoe", "watch"]

# effnetb2, effnetb2_transform = create_effnetb2_model(num_classes=len(class_names))
# effnetb2.load_state_dict(torch.load(f="effnetb2.pth",map_location=torch.device("cpu")))

# def predict(img) -> Tuple[Dict, float]:
#     """
#     Transforms and performs a prediction on img and returns prediction and time taken.
#     """
#     start_time = timer()

#     img = effnetb2_transform(img).unsqueeze(0)

#     effnetb2.eval()
#     with torch.inference_mode():
#         pred_probs = torch.softmax(effnetb2(img), dim=1)

#     pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}

#     pred_time = round(timer() - start_time, 5)

#     return pred_labels_and_probs, pred_time

# example_list = [["examples/" + example] for example in os.listdir("examples")]
# title = "Watch Shoe Fragrance Determinator!"
# description = "Using an EfficientNetB2 feature extraction model"
# demo = gr.Interface(
#     fn=predict,
#     inputs=gr.Image(type="pil"),
#     outputs=[gr.Label(num_top_classes=3, label="Predictions"), gr.Number(label="Prediction Time (s)")],
#     title=title, description=description,examples=example_list
# )
# demo.launch(debug=False, share=True)


In [17]:
print(torch.__version__)
print(torchvision.__version__)
print(gr.__version__)

2.11.0+cu128
0.26.0+cu128
6.26.0


In [18]:
# %%writefile demo/watch_shoe_fragrance/requirements.txt
# torch==2.11.0
# torchvision==0.26.0
# gradio==6.19.0

In [19]:
# !ls demo/watch_shoe_fragrance/

In [20]:
# !cd demo/watch_shoe_fragrance && zip -r ../wsf_huggingface.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"